## Task 2 Sentiment Analysis
I've used Kaggle IMDB Movie Review dataset

In [1]:
# Step 1: Import required libraries

import pandas as pd
import numpy as np
import re
import string

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
# Step 2: Load dataset

file_path = r"C:\Users\DELL\OneDrive\Desktop\ALL INTERNSHIP WORK\GEN_AI_MODULE\Task-2_Sentiment_Analysis\IMDB_Dataset.csv"

df = pd.read_csv(file_path)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# Dataset info

print("Shape of dataset:", df.shape)
df.info()

Shape of dataset: (50000, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [5]:
# Column names

print(df.columns)

Index(['review', 'sentiment'], dtype='object')


In [6]:
# Check target distribution

print(df['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [8]:
# Show some text samples

print(df['review'].head(5))

0    One of the other reviewers has mentioned that ...
1    A wonderful little production. <br /><br />The...
2    I thought this was a wonderful way to spend ti...
3    Basically there's a family where a little boy ...
4    Petter Mattei's "Love in the Time of Money" is...
Name: review, dtype: object


In [9]:
# Initialize NLP tools

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [10]:
# Text preprocessing function

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 3. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 4. Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # 5. Tokenization
    words = text.split()
    
    # 6. Remove stopwords & apply stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    
    # 7. Join back
    return " ".join(words)

In [11]:
# Apply preprocessing

df['cleaned_review'] = df['review'].apply(preprocess_text)

df[['review', 'cleaned_review']].head()

,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [12]:
# Check a few cleaned examples

for i in range(5):
    print("Original:", df['review'][i])
    print("Cleaned :", df['cleaned_review'][i])
    print("-" * 50)

Original: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due 

In [13]:
# Features and labels

X = df['cleaned_review']
y = df['sentiment']

In [14]:
# Split data

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (40000,)
Testing size: (10000,)


In [15]:
# Bag of Words

bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print("BoW shape:", X_train_bow.shape)

BoW shape: (40000, 120102)


In [16]:
# TF-IDF

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)

TF-IDF shape: (40000, 120102)


In [17]:
# Initialize models

lr = LogisticRegression()
nb = MultinomialNB()
dt = DecisionTreeClassifier()

In [18]:
# Train models using BoW

lr.fit(X_train_bow, y_train)
nb.fit(X_train_bow, y_train)
dt.fit(X_train_bow, y_train)

c:\Users\DELL\OneDrive\Desktop\ALL INTERNSHIP WORK\GEN_AI_MODULE\venv\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [19]:
# Predictions

y_pred_lr_bow = lr.predict(X_test_bow)
y_pred_nb_bow = nb.predict(X_test_bow)
y_pred_dt_bow = dt.predict(X_test_bow)

In [20]:
# Evaluation function

def evaluate_model(y_test, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))
    print()

In [21]:
# Evaluate BoW models

evaluate_model(y_test, y_pred_lr_bow, "Logistic Regression (BoW)")
evaluate_model(y_test, y_pred_nb_bow, "Naive Bayes (BoW)")
evaluate_model(y_test, y_pred_dt_bow, "Decision Tree (BoW)")

--- Logistic Regression (BoW) ---
Accuracy : 0.8813
Precision: 0.8814369906806936
Recall   : 0.8813
F1 Score : 0.8812785333488702

--- Naive Bayes (BoW) ---
Accuracy : 0.8554
Precision: 0.8558596904745922
Recall   : 0.8554
F1 Score : 0.8553766190060552

--- Decision Tree (BoW) ---
Accuracy : 0.7278
Precision: 0.7280625595628065
Recall   : 0.7278
F1 Score : 0.7277717986981362



In [22]:
# Train models using TF-IDF

lr_tfidf = LogisticRegression()
nb_tfidf = MultinomialNB()
dt_tfidf = DecisionTreeClassifier()

lr_tfidf.fit(X_train_tfidf, y_train)
nb_tfidf.fit(X_train_tfidf, y_train)
dt_tfidf.fit(X_train_tfidf, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [23]:
# Predictions

y_pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)
y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)
y_pred_dt_tfidf = dt_tfidf.predict(X_test_tfidf)

In [24]:
# Evaluate TF-IDF models

evaluate_model(y_test, y_pred_lr_tfidf, "Logistic Regression (TF-IDF)")
evaluate_model(y_test, y_pred_nb_tfidf, "Naive Bayes (TF-IDF)")
evaluate_model(y_test, y_pred_dt_tfidf, "Decision Tree (TF-IDF)")

--- Logistic Regression (TF-IDF) ---
Accuracy : 0.89
Precision: 0.8903348277941102
Recall   : 0.89
F1 Score : 0.8899615236460088

--- Naive Bayes (TF-IDF) ---
Accuracy : 0.8621
Precision: 0.8624910083600161
Recall   : 0.8621
F1 Score : 0.8620828951954893

--- Decision Tree (TF-IDF) ---
Accuracy : 0.7166
Precision: 0.7167933132955334
Recall   : 0.7166
F1 Score : 0.7165828595730845



### Model Comparison & Insights

1. Best Model:  
    Logistic Regression performed the best overall in both BoW and TF-IDF.
    It handled text data efficiently and gave stable results.

2. BoW vs TF-IDF:  
    TF-IDF performed better than Bag of Words in most cases.
    It captures importance of words rather than just frequency.

3. Naive Bayes:  
    Fast and efficient.
    Works well with text data but slightly lower accuracy than Logistic Regression.

4. Decision Tree:  
    Performed worse compared to other models.
    Tends to overfit on text data.

5. Conclusion:  
    Best combination: TF-IDF + Logistic Regression
    This combination gives better generalization and accuracy.